In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision  import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.optim as optim


In [ ]:
# Write your code here
### **🔹 General Structure of a Dataset Class**
from torch.utils.data import Dataset
from PIL import Image
import glob

class CustomDataset(Dataset):
    def __init__(self, root_dir,split="train", transform=None):    # Dont forget how to split between Train,val,test.
        self.root_dir = root_dir  # Dataset path
        self.transform = transform  # Transformations
        self.split=split
        self.class_labels = {"Potato___healthy": 0, "Potato___Late_blight": 1, "Potato___Early_blight": 2}

        # Get all image paths
        self.image_paths = []
        self.labels = []
        for class_name, label in self.class_labels.items():
            class_images = glob.glob(f"{root_dir}/PlantVillage/{self.split}/{class_name}/*.*")  # Find all images
            self.image_paths.extend(class_images)
            self.labels.extend([label] * len(class_images))  # Assign labels

    def __len__(self):
        return len(self.image_paths)  # Total number of images

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        label = self.labels[idx]  # Get label

        # Load image using PIL
        image = Image.open(image_path)

        # Apply transformations (if any)
        if self.transform:
            image = self.transform(image)

        return image, label  # Return processed image & label

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),  # Resize all images to 64x64
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # here 3 numbers because we have 3 channels
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)), # as of effecientNet
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = CustomDataset(path,"train",transform=transform_train)
test_dataset = CustomDataset(path,"test",transform=transform_test)
from torch.utils.data import DataLoader

# Define batch size
batch_size = 32

# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Get a batch of training images
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}, Labels: {labels}")

In [ ]:

# TO DO
import random
import numpy as np

# Create a function to visualize samples
def visualize_samples(dataset, num_samples=8,title="Dataset Samples",std=[0.229, 0.224, 0.225],mean=[0.485, 0.456, 0.406]):
    """
    Visualize random samples from a dataset.

    Args:
        dataset: PyTorch Dataset object
        num_samples: Number of samples to display
        title: Title for the plot
    """

    # Select random indices
    indices = random.sample(range(len(dataset)), num_samples)

    # Calculate grid size
    cols = 4
    rows = (num_samples + cols - 1) // cols

    # Create the plot
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        # Get image and label
        image, label = dataset[idx]
        # Convert tensor to numpy for display
        if isinstance(image, torch.Tensor):
            image = image.permute(1, 2, 0).numpy()  # CHW -> HWC
        # Get class name
        image = image*std + mean
        image=image.clip(0,1)               ##Focus ya dboj
        # Display image
        axes[i].imshow(image,cmap="gray")
        axes[i].set_title(f"(Label: {label})", fontsize=10)
        axes[i].axis('off')
    # Hide any unused subplots
    for i in range(num_samples, len(axes)):
        axes[i].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_samples(train_dataset,8,"Dataset Samples",std=[0.229, 0.224, 0.225],mean=[0.485, 0.456, 0.406])

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision  import datasets, transforms
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.optim as optim


In [ ]:
# Write your code here
class CustomModel(nn.Module):

  def __init__(self,num_classes=3):
    super().__init__()


    self.features = nn.Sequential(
        nn.Conv2d(3,16,kernel_size=3,padding=1,stride=1), #32x32x16
        nn.BatchNorm2d(16),
        nn.ReLU(),

        nn.Conv2d(16,32,kernel_size=3,padding=1,stride=1), #32
        nn.BatchNorm2d(32),
        nn.ReLU(),

        nn.Conv2d(32,64,kernel_size=3,padding=1,stride=1), #64
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.Conv2d(64,128,kernel_size=3,padding=1,stride=1), #64
        nn.BatchNorm2d(128),
        nn.MaxPool2d(kernel_size=2,stride=2), #16x16x128
        nn.ReLU(),

        nn.Conv2d(128,256,kernel_size=3,padding=1,stride=1), #64
        nn.BatchNorm2d(256),
        nn.MaxPool2d(kernel_size=2,stride=2), #8x8x128
        nn.ReLU(),

    )

    self.classifier = nn.Sequential(
        nn.Dropout(),
        nn.Flatten(),
        nn.Linear(8*8*256,128),
        nn.ReLU(),
        nn.Linear(128,num_classes)
    )
  def forward(self,x):
    x = self.features(x)
    x = self.classifier(x)
    return x


In [ ]:
# Write your code here
from tqdm import tqdm    # Shows progress bar

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)


        outputs = model(images)  # Forward pass               .squeeze() may solve problems**********
        loss = criterion(outputs, labels)  # Compute loss

        optimizer.zero_grad()  # Reset gradients
        loss.backward()  # Backpropagation
        optimizer.step()  # Update weights

        total_loss += loss.item()

        # Track accuracy
        outputs = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)  # Get class with highest probability
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)  # Forward pass               .squeeze() may solve problems**********
            loss = criterion(outputs, labels)  # Compute loss
            total_loss += loss.item()

            # Compute accuracy
            outputs = torch.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)  # Get predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage
    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
# Write your code here
class CustomModel2(nn.Module):

  def __init__(self,num_classes=3):
    super().__init__()


    self.layer1 = nn.Sequential(
        nn.Conv2d(3,16,kernel_size=3,padding=1,stride=1), #32x32x16
        nn.BatchNorm2d(16),
        nn.ReLU(),
    )
    self.layer2 = nn.Sequential(
        nn.Conv2d(16,32,kernel_size=3,padding=1,stride=1), #32x32x32
        nn.BatchNorm2d(32),
        nn.ReLU(),
    )
    self.layer3 = nn.Sequential(
        nn.Conv2d(32,64,kernel_size=3,padding=1,stride=1), #32x32x64
        nn.BatchNorm2d(64),
        nn.ReLU(),
    )
    self.layer4 = nn.Sequential(
        nn.Conv2d(64,128,kernel_size=3,padding=1,stride=1), #32x32x128
        nn.BatchNorm2d(128),
        nn.ReLU(),
    )
    self.layer5 = nn.Sequential(
        nn.Conv2d(160,256,kernel_size=3,padding=1,stride=1), #64
        nn.BatchNorm2d(256),
        nn.MaxPool2d(kernel_size=2,stride=2), #16x16x256
        nn.ReLU(),
    )

    self.classifier = nn.Sequential(
        nn.Dropout(),
        nn.Flatten(),
        nn.Linear(16*16*256,128),
        nn.ReLU(),
        nn.Linear(128,num_classes)
    )
  def forward(self,x):
    x = self.layer1(x)
    res2 = self.layer2(x)
    x = self.layer3(res2)
    x = self.layer4(x)
    x=torch.cat([x , res2],dim=1)
    x = self.layer5(x)
    x = self.classifier(x)
    return x


In [ ]:
# Write your code here
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel2().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10 # Number of epochs


# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate(model, test_loader, criterion, device)

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()